In [ ]:
'''
Train a Transformer encoder model to generate MIDI in two parts:
    1. pre-train
    2. SFT
'''
from pathlib import Path
from random import shuffle

from miditok import REMI, TokenizerConfig
from miditok.utils import split_files_for_training

from config import ModelParams, HyperParams

PREPROCESS = False # set to True if you haven't run preprocess yet
TRAIN = True # set to False if you want to skip training

TOKENIZER_PARAMS = {
    "special_tokens": ["PAD_None", "MASK_None"],
    "pitch_range": (22, 107),
    "num_velocities": 24,
    "beat_res": {(0, 8): 4}, # 16 Position tokens within a bar
    "use_pitchdrum_tokens": False,
    "use_chords": False,
    "chord_tokens_with_root_note": False,
}
config = TokenizerConfig(**TOKENIZER_PARAMS)
tokenizer = REMI(config)

if PREPROCESS:
    midi_paths = list(Path(HyperParams.DATA_PATH).rglob("*.mid"))
    shuffle(midi_paths)
    midi_paths = [p.resolve() for p in midi_paths if p.is_file()]
    total_num_files = len(midi_paths)

    tokenizer.save(Path("models", HyperParams.name + "_tokenizer.json"))

    num_files_valid = round(total_num_files * 0.03)
    num_files_test = round(total_num_files * 0.03)
    midi_paths_val = midi_paths[:num_files_valid]
    midi_paths_test = midi_paths[num_files_valid:num_files_valid + num_files_test]
    midi_paths_train = midi_paths[num_files_valid + num_files_test:]
    
    # Chunk MIDIs and perform data augmentation on each subset independently
    for files_paths, subset_name in (
        (midi_paths_train, "train"), (midi_paths_val, "val"), (midi_paths_test, "test")
    ):
        subset_chunks_dir = Path(f"dataset/{subset_name}")
        split_files_for_training(
            files_paths=files_paths,
            tokenizer=tokenizer,
            save_dir=subset_chunks_dir,
            max_seq_len=1024,
            num_overlap_bars=0,
        )
else:
    tokenizer._load_from_json(Path("models", HyperParams.name + "_tokenizer.json"))

print(tokenizer.vocab)
print(tokenizer.vocab_size)

In [ ]:
from help import MIDIDataset, DataCollator
from torch.utils.data import DataLoader

midi_paths_train = list(Path("dataset", "train").rglob("*.mid"))
midi_paths_val = list(Path("dataset", "val").rglob("*.mid"))

train_dataset = MIDIDataset(midi_paths_train, tokenizer)
val_dataset = MIDIDataset(midi_paths_val, tokenizer)

# Extend sequences with EOS tokens (EOS tokens are treated as normal tokens during training)
collator = DataCollator(pad_token_id=tokenizer.vocab['PAD_None'])

train_dataloader = DataLoader(dataset=train_dataset, batch_size=HyperParams.batch_size, collate_fn=collator, shuffle=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=HyperParams.batch_size, collate_fn=collator, shuffle=True)

'''
for batch in train_dataloader:
    print(batch['input_ids'].shape)
    print(batch['prompt_lengths'])
    
    test = tokenizer.decode(batch['input_ids'][0].unsqueeze(0))
    print(test)
    test.dump_midi(Path("midi_start.mid"))
    break
'''

In [ ]:
from help import Trainer
import torch
from tqdm.notebook import tqdm

trainer = Trainer(ModelParams, HyperParams, tokenizer)

print(sum(p.numel() for p in trainer.mask_predictor.parameters()), "model parameters\n")
print("Model architecture:")
print(trainer.mask_predictor)

In [ ]:
if TRAIN:
    from torch.utils.tensorboard import SummaryWriter
    from itertools import cycle
    # Setup
    trainer.phase = 'pre-training' # 'pre-training' or 'SFT'
    model_dir = 'models/checkpoints/' + trainer.phase + '_'

    writer = SummaryWriter()
    global_step = 0

    # Optional: wrap dataloader with cycle to avoid worrying about epoch ends
    train_iterator = cycle(train_dataloader)

    pbar = tqdm(total=HyperParams.num_training_steps, desc="Training", position=0)

    best_val = 1e8
    train_loss = 0

    if HyperParams.retrain:
        loss = trainer.load("models/checkpoints/run_0/pre-training_400000_checkpoint.pth")

    trainer.mask_predictor.train()
    while global_step < HyperParams.num_training_steps:
        batch = next(train_iterator)
        input_ids = batch['input_ids'].to(trainer.device)
        prompt_lengths = batch['prompt_lengths'].to(trainer.device)

        train_loss += trainer.train_step(input_ids, prompt_lengths)  

        # Logging
        global_step += 1
        if global_step % HyperParams.logging_interval == 0:
            trainer.mask_predictor.eval()

            val_loss = 0
            for batch in tqdm(val_dataloader, desc="Validating", leave=False):
                input_ids = batch['input_ids'].to(trainer.device)
                prompt_lengths = batch['prompt_lengths'].to(trainer.device)
                val_loss += trainer.val_step(input_ids, prompt_lengths)

            if val_loss < best_val:
                trainer.save('models/best_val', global_step, val_loss / len(val_dataloader))
                best_val = val_loss

            writer.add_scalar('Loss/val', val_loss / len(val_dataloader), global_step)       
            writer.add_scalar('Loss/train', train_loss / HyperParams.logging_interval, global_step)
            writer.add_scalar('LR', trainer.lr_scheduler.get_last_lr()[0], global_step)
            train_loss = 0

            trainer.save(model_dir + str(global_step), global_step, val_loss)

            trainer.mask_predictor.train()
        pbar.update(1)

    pbar.close()
    writer.close()

In [ ]:
import numpy as np
import torch.nn.functional as F

# https://github.com/ML-GSAI/LLaDA
def add_gumbel_noise(logits, temperature):
    '''
    The Gumbel max is a method for sampling categorical distributions.
    According to arXiv:2409.02908, for MDM, low-precision Gumbel Max improves perplexity score but reduces generation quality.
    Thus, we use float64.
    '''
    if temperature == 0:
        return logits
    logits = logits.to(torch.float64)
    noise = torch.rand_like(logits, dtype=torch.float64)
    gumbel_noise = (- torch.log(noise)) ** temperature
    return logits.exp() / gumbel_noise

# https://github.com/ML-GSAI/LLaDA
def get_num_transfer_tokens(mask_index, steps):
    '''
    In the reverse process, the interval [0, 1] is uniformly discretized into steps intervals.
    Furthermore, because LLaDA employs a linear noise schedule (as defined in Eq. (8)),
    the expected number of tokens transitioned at each step should be consistent.

    This function is designed to precompute the number of tokens that need to be transitioned at each step.
    '''
    mask_num = mask_index.sum(dim=1, keepdim=True)

    base = mask_num // steps
    remainder = mask_num % steps

    num_transfer_tokens = torch.zeros(mask_num.size(0), steps, device=mask_index.device, dtype=torch.int64) + base

    for i in range(mask_num.size(0)):
        num_transfer_tokens[i, :remainder[i]] += 1

    return num_transfer_tokens

# https://github.com/ML-GSAI/LLaDA
@ torch.no_grad()
def generate(model, prompt, steps=512, gen_length=512, block_length=512, temperature=0.,
             cfg_scale=0., remasking='low_confidence', mask_id=tokenizer['MASK_None']):
    '''
    Args:
        model: Mask predictor.
        prompt: A tensor of shape (1, L).
        steps: Sampling steps, less than or equal to gen_length.
        gen_length: Generated answer length.
        block_length: Block length, less than or equal to gen_length. If less than gen_length, it means using semi_autoregressive remasking.
        temperature: Categorical distribution sampling temperature.
        cfg_scale: Unsupervised classifier-free guidance scale.
        remasking: Remasking strategy. 'low_confidence' or 'random'.
        mask_id: The toke id of [MASK] is tokenizer['MASK_None'].
    '''
    x = torch.full((1, prompt.shape[1] + gen_length), mask_id, dtype=torch.long).to(prompt.device)
    x[:, :prompt.shape[1]] = prompt.clone()

    prompt_index = (x != mask_id)

    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length

    assert steps % num_blocks == 0
    steps = steps // num_blocks

    for num_block in range(num_blocks):
        block_mask_index = (x[:, prompt.shape[1] + num_block * block_length: prompt.shape[1] + (num_block + 1) * block_length:] == mask_id)
        num_transfer_tokens = get_num_transfer_tokens(block_mask_index, steps)
        for i in range(steps):
            mask_index = (x == mask_id)
            if cfg_scale > 0.:
                un_x = x.clone()
                un_x[prompt_index] = mask_id
                x_ = torch.cat([x, un_x], dim=0)
                logits = model(x_).logits
                logits, un_logits = torch.chunk(logits, 2, dim=0)
                logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
            else:
                logits = model(x)

            logits_with_noise = add_gumbel_noise(logits, temperature=temperature)
            x0 = torch.argmax(logits_with_noise, dim=-1) # b, l

            if remasking == 'low_confidence':
                p = F.softmax(logits, dim=-1)
                x0_p = torch.squeeze(
                    torch.gather(p, dim=-1, index=torch.unsqueeze(x0, -1)), -1) # b, l
            elif remasking == 'random':
                x0_p = torch.rand((x0.shape[0], x0.shape[1]), device=x0.device)
            else:
                raise NotImplementedError(remasking)

            x0_p[:, prompt.shape[1] + (num_block + 1) * block_length:] = -np.inf

            x0 = torch.where(mask_index, x0, x)
            confidence = torch.where(mask_index, x0_p, -np.inf)

            transfer_index = torch.zeros_like(x0, dtype=torch.bool, device=x0.device)
            for j in range(confidence.shape[0]):
                _, select_index = torch.topk(confidence[j], k=num_transfer_tokens[j, i])
                transfer_index[j, select_index] = True
            x[transfer_index] = x0[transfer_index]

    return x

In [ ]:
# Generate MIDI
import random

device = 'cuda'

tokenizer = REMI(params=Path("models", HyperParams.name + "_tokenizer.json"))
trainer = Trainer(ModelParams, HyperParams, tokenizer)
trainer.load('models/checkpoints/pre-training_500_checkpoint.pth')

midi_paths_test = list(Path("dataset", "test").rglob("*.mid"))
index = random.randint(0, len(midi_paths_test) - 1)
midi = tokenizer(midi_paths_test[index])[0] #TokSequence

gen_length = 512
prompt_length = 512
prompt = torch.tensor(midi.ids[:prompt_length]).unsqueeze(0).to(device)
gen_answer = generate(trainer.mask_predictor, prompt, gen_length=gen_length, block_length=gen_length)[:, prompt_length:]

# generate MIDI
gen_answer_midi = tokenizer(gen_answer.tolist())
prompt_midi = tokenizer(prompt.tolist())
answer_midi = tokenizer([midi.ids[512:1024]])

gen_answer_midi.dump_midi(Path("midi_gen_answer.mid"))
prompt_midi.dump_midi(Path("midi_prompt.mid"))
answer_midi.dump_midi(Path("midi_answer.mid"))